## 7. Class Imbalance Handling--------------------------------------------------

# BALANCED RANDOM FOREST

In [ ]:
# Balanced Random Forest to handles imbalance internally
# At each bootstrap sample, it undersamples the majority class to match minority class size
# No need for SMOTE instead uses sklearn Pipeline (not imblearn)
brf_pipe = SkPipeline([
    ('preprocess', preprocess),
    ('clf', BalancedRandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

brf_params = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [10, 20, None],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf': [1, 2],
    'clf__max_features': ['sqrt', 'log2'],
    'clf__sampling_strategy': ['auto', 'not majority']  # how to resample
}

# using RandomizedSearchCV for tuning
brf_search = RandomizedSearchCV(
    brf_pipe, param_distributions=brf_params,
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
brf_search.fit(X_train, y_train)
brf_best = brf_search.best_estimator_

print('Best params (BRF):', brf_search.best_params_)
print(f'Best CV F1 Macro: {brf_search.best_score_:.4f}')
brf_metrics = evaluate('Balanced Random Forest \u2014 VALIDATION', brf_best, X_val, y_val)
results_table.append({'Model': 'Balanced Random Forest', **brf_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params (BRF): {'clf__sampling_strategy': 'auto', 'clf__n_estimators': 300, 'clf__min_samples_split': 2, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': None}
Best CV F1 Macro: 0.7165

=== Balanced Random Forest — VALIDATION ===
              precision    recall  f1-score   support

           1      0.808     0.804     0.806     42368
           2      0.881     0.747     0.809     56660
           3      0.801     0.852     0.826      7151
           4      0.537     0.973     0.692       549
           5      0.316     0.968     0.476      1899
           6      0.573     0.897     0.699      3474
           7      0.649     0.980     0.781      4102

    accuracy                          0.792    116203
   macro avg      0.652     0.889     0.727    116203
weighted avg      0.822     0.792     0.799    116203

Accuracy         : 0.7915
Balanced Accuracy: 0.8887
F1 Macro         : 0.7270

Pe